# Wine Dataset Demo

This notebook demonstrates the core dependencies in `requirements.txt` using the Wine dataset from scikit-learn.

In [ ]:
from __future__ import annotations

from io import StringIO
from os import getenv

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field, computed_field

## Fetch the Dataset

`requests` retrieves the raw data from scikit-learn.

In [ ]:
load_dotenv()

DATA_URL = "https://raw.githubusercontent.com/scikit-learn/scikit-learn/main/sklearn/datasets/data/wine_data.csv"
github_token = getenv("GH_TOKEN") or getenv("GITHUB_TOKEN")
headers = {"Authorization": f"Bearer {github_token}"} if github_token else {}

data_response = requests.get(DATA_URL, headers=headers, timeout=10)
data_response.raise_for_status()

## Parse the Header

The first row is not a normal tabular header. It stores `n_samples`, `n_features`, and target names. The actual observations begin on the second row.

In [ ]:
feature_names = [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280/od315_of_diluted_wines",
    "proline",
]

lines = data_response.text.splitlines()
metadata = lines[0].split(",")
expected_samples = int(metadata[0])
expected_features = int(metadata[1])
target_names = metadata[2:]

df = pd.read_csv(
    StringIO("\n".join(lines[1:])),
    header=None,
    names=[*feature_names, "target"],
).convert_dtypes(dtype_backend="pyarrow")
df["wine_class"] = df["target"].map(dict(enumerate(target_names)))

assert len(df) == expected_samples
assert len(feature_names) == expected_features

df.head()

## Validate Rows with Pydantic

A typed model gives an explicit contract for individual observations. Here it also computes a compact ratio that is useful for quick comparisons between classes.

In [ ]:
class WineSample(BaseModel):
    model_config = ConfigDict(extra="forbid")

    alcohol: float = Field(gt=0)
    malic_acid: float = Field(gt=0)
    ash: float = Field(gt=0)
    alcalinity_of_ash: float = Field(gt=0)
    magnesium: float = Field(gt=0)
    total_phenols: float = Field(gt=0)
    flavanoids: float = Field(ge=0)
    nonflavanoid_phenols: float = Field(ge=0)
    proanthocyanins: float = Field(ge=0)
    color_intensity: float = Field(gt=0)
    hue: float = Field(gt=0)
    od280_od315_of_diluted_wines: float = Field(gt=0, alias="od280/od315_of_diluted_wines")
    proline: float = Field(gt=0)
    target: int = Field(ge=0, le=2)
    wine_class: str

    @computed_field
    @property
    def phenol_to_flavanoid_ratio(self) -> float:
        return round(self.total_phenols / self.flavanoids, 3)


validated_samples = [WineSample.model_validate(row) for row in df.head(5).to_dict(orient="records")]
validated_samples[0]

## Summarize with Pandas

The dataframe uses Arrow-backed nullable dtypes via `pandas[pyarrow]`, then groups by class to compare the most interpretable chemical measurements.

In [ ]:
profile_columns = ["alcohol", "malic_acid", "total_phenols", "flavanoids", "color_intensity", "proline"]

class_profile = df.groupby("wine_class", observed=True)[profile_columns].mean().sort_values("alcohol", ascending=False)
class_profile.round(2)

## Plot Class Differences

Pandas plotting is enough for quick visual checks. Standardizing each feature by its maximum keeps features with different units readable on one chart.

In [ ]:
scaled_profile = class_profile / class_profile.max()

ax = scaled_profile.T.plot(kind="bar", figsize=(10, 4), title="Relative Mean Feature Values by Wine Class")
ax.set_xlabel("Feature")
ax.set_ylabel("Mean divided by feature maximum")
ax.legend(title="Wine class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.show()

## Explore Interactively

Use widgets to compare any two chemical measurements and optionally filter to one target class.

In [ ]:
class_options = ["All", *target_names]

x_picker = widgets.Dropdown(options=feature_names, value="alcohol", description="X")
y_picker = widgets.Dropdown(options=feature_names, value="flavanoids", description="Y")
class_picker = widgets.Dropdown(options=class_options, value="All", description="Class")


def plot_features(x: str, y: str, wine_class: str) -> None:
    view = df if wine_class == "All" else df.loc[df["wine_class"] == wine_class]
    ax = view.plot.scatter(x=x, y=y, c="target", colormap="viridis", figsize=(6, 4), title=f"{y} vs {x}")
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.show()


controls = widgets.HBox([x_picker, y_picker, class_picker])
interactive = widgets.interactive_output(
    plot_features,
    {"x": x_picker, "y": y_picker, "wine_class": class_picker},
)

display(controls, interactive)